In [ ]:
from datasets import load_dataset
import pandas as pd
dataset = load_dataset("electricsheepafrica/nigerian-banking-bnpl")
df = dataset["train"].to_pandas()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.head()

In this notebook we have two main and two side predictions. This is a mix of two regression and two classification problems:

Main questions:
1) Classification: Can we predict whether a transaction will default within 90 days?
2) Regression: Can we predict a customer's credit score from origination information?

Side questions:

3) Classification: Can we predict whether a transaction will default within 30 days?

4) Regression: Can we predict the principal amount (principal_ngn)?


Some important things that should be mentioned:
- The data is split by time using `purchase_date`: oldest 70% train, next 15% validation, newest 15% test
- Cross-validation is done only on the training set using 5-fold `TimeSeriesSplit`
- `transaction_id` and `customer_id` are not used as model features
- `merchant_name` is dropped because it can have many unique values and may make the notebook harder to explain
- `first_time_customer` is encoded using `OneHotEncoder()` mainly for practice and experimentation

# Problem 1

## Question
Can we predict whether a transaction will default within **90 days** using information available around loan origination?

## Target
`default_90d`

## Model
**Logistic Regression**

## Why this model?
Logistic Regression is a good baseline classification model. Super easy to explain. Works well with scaled numeric features and encoded categorical features. Also gives predicted probabilities that can be used for ROC/AUC.

## Evaluation metrics
For classification, this notebook uses Accuracy, Precision, Recall, F1 score, Confusion Matrix, and ROC/AUC.


## Feature engineering
This main model uses engineered date features:
- purchase month
- purchase quarter
- purchase day of week
- weekend purchase flag
- days until first payment


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, RocCurveDisplay
)

#Make a separate copy of the dataframe for this problem
problem1_df = df.copy()

#Convert date columns
problem1_df["purchase_date"] = pd.to_datetime(problem1_df["purchase_date"], errors="coerce")
problem1_df["first_payment_due"] = pd.to_datetime(problem1_df["first_payment_due"], errors="coerce")

#Sort by time so the train/validation/test split is realistic
problem1_df = problem1_df.sort_values("purchase_date").reset_index(drop=True)

#Engineered date features
problem1_df["purchase_month"] = problem1_df["purchase_date"].dt.month
problem1_df["purchase_quarter"] = problem1_df["purchase_date"].dt.quarter
problem1_df["purchase_dayofweek"] = problem1_df["purchase_date"].dt.dayofweek
problem1_df["is_weekend"] = problem1_df["purchase_dayofweek"].isin([5, 6]).astype(int)
problem1_df["days_until_first_payment"] = (
    problem1_df["first_payment_due"] - problem1_df["purchase_date"]
).dt.days

#Drop rows where the target is missing
problem1_df = problem1_df.dropna(subset=["default_90d"])

#Convert boolean target to 0/1
problem1_df["default_90d"] = problem1_df["default_90d"].astype(int)

#Feature columns for this model
#We don't use transaction_id, customer_id, merchant_name, default_30d, or default_90d as features.
problem1_numeric_features = [
    "principal_ngn", "interest_rate_monthly", "tenor_days", "num_installments",
    "credit_score", "purchase_month", "purchase_quarter", "purchase_dayofweek",
    "is_weekend", "days_until_first_payment"
]

problem1_categorical_features = [
    "merchant_category", "customer_state", "provider", "first_time_customer"
]

problem1_features = problem1_numeric_features + problem1_categorical_features

X_problem1 = problem1_df[problem1_features]
y_problem1 = problem1_df["default_90d"]

print("Problem 1 feature columns:")
print(problem1_features)
print("\nTarget balance:")
print(y_problem1.value_counts(normalize=True))

In [ ]:
# Time-based train/validation/test split
n_rows = len(problem1_df)
train_end = int(n_rows * 0.70)
val_end = int(n_rows * 0.85)

X1_train = X_problem1.iloc[:train_end]
y1_train = y_problem1.iloc[:train_end]

X1_val = X_problem1.iloc[train_end:val_end]
y1_val = y_problem1.iloc[train_end:val_end]

X1_test = X_problem1.iloc[val_end:]
y1_test = y_problem1.iloc[val_end:]

print("Train shape:", X1_train.shape)
print("Validation shape:", X1_val.shape)
print("Test shape:", X1_test.shape)

In [ ]:
#Preprocessing for numeric and categorical columns
problem1_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

problem1_categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    # OneHotEncoder is used on first_time_customer mainly for practice and experimentation. We know it isn't necesary
    # A boolean column could also be converted to 0/1, but using OneHotEncoder helps practice sklearn preprocessing.
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

problem1_preprocessor = ColumnTransformer(transformers=[
    ("num", problem1_numeric_transformer, problem1_numeric_features),
    ("cat", problem1_categorical_transformer, problem1_categorical_features)
], sparse_threshold=0)

problem1_model = Pipeline(steps=[
    ("preprocessor", problem1_preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

#5-fold cross-validation on the training set only
problem1_cv = TimeSeriesSplit(n_splits=5)
problem1_scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

problem1_cv_results = cross_validate(
    problem1_model,
    X1_train,
    y1_train,
    cv=problem1_cv,
    scoring=problem1_scoring,
    n_jobs=-1
)

problem1_cv_summary = pd.DataFrame(problem1_cv_results).mean().to_frame("Mean CV Score")
display(problem1_cv_summary)

In [ ]:
#Fitting the model on the training data
problem1_model.fit(X1_train, y1_train)

#Validation predictions
problem1_val_pred = problem1_model.predict(X1_val)
problem1_val_proba = problem1_model.predict_proba(X1_val)[:, 1]

#Test predictions
problem1_test_pred = problem1_model.predict(X1_test)
problem1_test_proba = problem1_model.predict_proba(X1_test)[:, 1]

#Validation and test metrics
problem1_results = pd.DataFrame({
    "Dataset": ["Validation", "Test"],
    "Accuracy": [
        accuracy_score(y1_val, problem1_val_pred),
        accuracy_score(y1_test, problem1_test_pred)
    ],
    "Precision": [
        precision_score(y1_val, problem1_val_pred, zero_division=0),
        precision_score(y1_test, problem1_test_pred, zero_division=0)
    ],
    "Recall": [
        recall_score(y1_val, problem1_val_pred, zero_division=0),
        recall_score(y1_test, problem1_test_pred, zero_division=0)
    ],
    "F1": [
        f1_score(y1_val, problem1_val_pred, zero_division=0),
        f1_score(y1_test, problem1_test_pred, zero_division=0)
    ],
    "ROC_AUC": [
        roc_auc_score(y1_val, problem1_val_proba),
        roc_auc_score(y1_test, problem1_test_proba)
    ]
})

display(problem1_results)

In [ ]:
#Confusion matrix on test set
problem1_cm = confusion_matrix(y1_test, problem1_test_pred)
ConfusionMatrixDisplay(problem1_cm).plot()
plt.title("Problem 1 Test Confusion Matrix: Logistic Regression")
plt.show()

# ROC curve on test set
RocCurveDisplay.from_predictions(y1_test, problem1_test_proba)
plt.title("Problem 1 Test ROC Curve: Logistic Regression")
plt.show()

#Simple train vs validation to check for overfitting
problem1_train_pred = problem1_model.predict(X1_train)
problem1_train_proba = problem1_model.predict_proba(X1_train)[:, 1]

problem1_overfit_check = pd.DataFrame({
    "Dataset": ["Train", "Validation"],
    "Accuracy": [accuracy_score(y1_train, problem1_train_pred), accuracy_score(y1_val, problem1_val_pred)],
    "F1": [f1_score(y1_train, problem1_train_pred, zero_division=0), f1_score(y1_val, problem1_val_pred, zero_division=0)],
    "ROC_AUC": [roc_auc_score(y1_train, problem1_train_proba), roc_auc_score(y1_val, problem1_val_proba)]
})

display(problem1_overfit_check)
problem1_overfit_check.set_index("Dataset")[["Accuracy", "F1", "ROC_AUC"]].plot(kind="bar")
plt.title("Problem 1 Train vs Validation Metrics")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.show()